## Load PLINK epistasis output (LD-pruned × QTL)

We load:
- `*.epi.qt` : pairwise interaction test results
- `*.epi.qt.summary` : per-set-SNP counts and best partner SNP

Because the analysis depends on the test mode, we parameterize:
- DRUG (PULV/FLU)
- RUN_LABEL (ldpruned_x_qtl)
and then reconstruct the QTL SNP set from the QTL interval file to classify each interaction as QTL vs non-QTL.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DRUG = "FLU"                 # "FLU" or "PULV"
RUN_LABEL = "ldpruned_x_qtl"  # for this run

BASE = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis")
DATA_DIR = BASE / "data"

# Epistasis outputs from 03
EPI_QT = DATA_DIR / f"epi_{DRUG.lower()}_{RUN_LABEL}.epi.qt"
EPI_SUM = DATA_DIR / f"epi_{DRUG.lower()}_{RUN_LABEL}.epi.qt.summary"

# Inputs needed to reconstruct QTL SNPs
SNP_INFO_TSV = DATA_DIR / "snp_info_complete.tsv"
QTL_RESULTS_TSV = {
    "FLU":  DATA_DIR / "Fluconazole_QTL_genes_20260211.csv",
    "PULV": DATA_DIR / "Pulvinatal_QTL_genes_20260210.csv",
}[DRUG]

for p in [EPI_QT, EPI_SUM, SNP_INFO_TSV, QTL_RESULTS_TSV]:
    if not p.exists():
        raise FileNotFoundError(p)

print("EPI_QT:", EPI_QT)
print("EPI_SUM:", EPI_SUM)
print("QTL_RESULTS_TSV:", QTL_RESULTS_TSV)

EPI_QT: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/epi_pulv_ldpruned_x_qtl.epi.qt
EPI_SUM: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/epi_pulv_ldpruned_x_qtl.epi.qt.summary
QTL_RESULTS_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Pulvinatal_QTL_genes_20260210.csv


## Read PLINK outputs and compute Bonferroni threshold

Bonferroni uses the total number of tests reported in the summary file.
We then subset `epi` to genome-wide significant interactions.

In [2]:
epi = pd.read_csv(EPI_QT, delim_whitespace=True)
summary = pd.read_csv(EPI_SUM, delim_whitespace=True)

print("epi shape:", epi.shape)
print("summary shape:", summary.shape)

total_tests = int(summary["N_TOT"].sum())
bonf = 0.05 / total_tests
print("Total tests:", total_tests)
print("Bonferroni:", bonf)

epi_sig = epi[epi["P"] < bonf].copy()
print("Genome-wide significant interactions:", epi_sig.shape[0])
print("Min P:", epi_sig["P"].min())

epi shape: (35453, 7)
summary shape: (1475, 8)
Total tests: 4628241
Bonferroni: 1.080324036712868e-08


/scratch/local/24796790/ipykernel_1779143/4122568296.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  epi = pd.read_csv(EPI_QT, delim_whitespace=True)
/scratch/local/24796790/ipykernel_1779143/4122568296.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  summary = pd.read_csv(EPI_SUM, delim_whitespace=True)


Genome-wide significant interactions: 11808
Min P: 3.436e-126


## Reconstruct QTL SNP set from interval file

The QTL results file contains intervals (chrom/start/end), not SNP IDs.
We intersect intervals with `snp_info_complete.tsv` to obtain the set of QTL SNPs.

In [3]:
qtl_df = pd.read_csv(QTL_RESULTS_TSV, sep=None, engine="python")
required = {"interval_id", "chrom", "start", "end"}
missing = required - set(qtl_df.columns)
if missing:
    raise ValueError(f"Missing columns in QTL file: {missing}. Found: {list(qtl_df.columns)}")

qtl_df["chrom"] = qtl_df["chrom"].astype(int)
qtl_df["start"] = qtl_df["start"].astype(int)
qtl_df["end"]   = qtl_df["end"].astype(int)

snp_info = pd.read_csv(SNP_INFO_TSV, sep="\t")
for c in ["SNP", "Chromosome", "Position (bp)"]:
    if c not in snp_info.columns:
        raise ValueError(f"snp_info missing column {c}")

qtl_snps = set()
for _, r in qtl_df.iterrows():
    hits = snp_info.loc[
        (snp_info["Chromosome"] == r["chrom"]) &
        (snp_info["Position (bp)"] >= r["start"]) &
        (snp_info["Position (bp)"] <= r["end"]),
        "SNP"
    ].astype(str).str.strip()
    qtl_snps.update(hits.tolist())

qtl_snps = {s for s in qtl_snps if s and s.lower() != "nan"}
print("Total QTL SNPs reconstructed:", len(qtl_snps))

Total QTL SNPs reconstructed: 3141


## Classify pairs and keep only QTL × non-QTL interactions

In LD-pruned × QTL mode, PLINK tests across two sets.
However, if some QTL SNPs are also in the LD-pruned set, QTL×QTL can appear.
We enforce that each retained interaction contains exactly one QTL SNP.

In [4]:
epi_sig["SNP1_is_qtl"] = epi_sig["SNP1"].isin(qtl_snps)
epi_sig["SNP2_is_qtl"] = epi_sig["SNP2"].isin(qtl_snps)

print("Proportion exactly one QTL:", (epi_sig["SNP1_is_qtl"] ^ epi_sig["SNP2_is_qtl"]).mean())
print("QTL×QTL:", (epi_sig["SNP1_is_qtl"] & epi_sig["SNP2_is_qtl"]).sum())
print("Neither QTL:", (~epi_sig["SNP1_is_qtl"] & ~epi_sig["SNP2_is_qtl"]).sum())

epi_clean = epi_sig[epi_sig["SNP1_is_qtl"] ^ epi_sig["SNP2_is_qtl"]].copy()

epi_clean["QTL_SNP"] = np.where(epi_clean["SNP1_is_qtl"], epi_clean["SNP1"], epi_clean["SNP2"])
epi_clean["MOD_SNP"] = np.where(epi_clean["SNP1_is_qtl"], epi_clean["SNP2"], epi_clean["SNP1"])

print("Retained QTL×LD interactions:", epi_clean.shape[0])
print("Unique QTL SNPs involved:", epi_clean["QTL_SNP"].nunique())
print("Unique modifier SNPs involved:", epi_clean["MOD_SNP"].nunique())

Proportion exactly one QTL: 0.9468157181571816
QTL×QTL: 628
Neither QTL: 0
Retained QTL×LD interactions: 11180
Unique QTL SNPs involved: 512
Unique modifier SNPs involved: 162


## Summarize the LD-pruned epistasis results at a glance

After genome-wide filtering and enforcing **exactly one QTL SNP per interaction**:

- **Proportion with exactly one QTL SNP:** 0.9468  
- **QTL × QTL interactions removed:** 628  
- **QTL × LD-pruned interactions retained:** 11,180  
- **Unique QTL SNPs involved:** 512  
- **Unique modifier (LD-pruned) SNPs involved:** 162  

Next, we’ll identify the **top modifier SNPs** by:
- how many significant interactions they participate in
- their strongest (minimum) P-value
- their largest |β_int|

In [5]:
# Code Cell — Top modifier SNPs (LD-pruned side) driving significant interactions

mod_strength = (
    epi_clean
    .groupby("MOD_SNP")
    .agg(
        n_interactions=("P", "count"),
        min_p=("P", "min"),
        max_abs_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values(["n_interactions", "min_p"], ascending=[False, True])
)

mod_strength.head(20)

,n_interactions,min_p,max_abs_beta
MOD_SNP,,,
snp34749,458,1.475000e-29,0.018571
snp34745,257,9.999000e-31,0.019578
snp14467,127,3.436000e-126,0.032397
snp14409,127,1.228000e-122,0.031940
snp14623,127,3.256000e-109,0.030293
snp14370,127,6.232000e-108,0.030114
snp14349,127,4.818000e-102,0.029394
snp14677,127,2.861000e-99,0.028988
snp14697,127,4.475000e-93,0.028196


## Localize Top Modifier SNPs (LD-pruned analysis)

We now map the top modifier SNPs to:
- Chromosome
- Genomic position

This tells us whether LD-pruned analysis recovers the same major modifier loci
seen in the genome-wide (all×QTL) scan.

In [6]:
# Take top 20 modifier SNPs
top_mod_snps = mod_strength.head(20).reset_index()

# Merge genomic position
top_mod_snps = top_mod_snps.merge(
    snp_info[["SNP", "Chromosome", "Position (bp)"]],
    left_on="MOD_SNP",
    right_on="SNP",
    how="left"
).drop(columns=["SNP"])

top_mod_snps.sort_values(["Chromosome", "Position (bp)"])

,MOD_SNP,n_interactions,min_p,max_abs_beta,Chromosome,Position (bp)
17,snp14269,127,1.930000e-56,0.022325,7,425446
15,snp14274,127,1.861000e-60,0.023040,7,430559
14,snp14304,127,4.313000e-70,0.024724,7,436337
12,snp14310,127,5.776000e-74,0.025363,7,442184
11,snp14319,127,2.210000e-81,0.026524,7,446205
9,snp14330,127,7.472000e-91,0.027866,7,449637
6,snp14349,127,4.818000e-102,0.029394,7,453554
5,snp14370,127,6.232000e-108,0.030114,7,458281
3,snp14409,127,1.228000e-122,0.031940,7,465447
2,snp14467,127,3.436000e-126,0.032397,7,479589


## Cluster LD-pruned Modifier SNPs into Independent Loci (10kb window)

We cluster modifier SNPs within 10kb
to estimate the number of independent modifier loci
in the LD-pruned analysis.

In [7]:
WINDOW = 10000

mod_pos = top_mod_snps[["MOD_SNP", "Chromosome", "Position (bp)"]].copy()
mod_pos = mod_pos.sort_values(["Chromosome", "Position (bp)"])

loci = []
current_chr = None
start = None
end = None

for _, row in mod_pos.iterrows():
    chr_ = row["Chromosome"]
    pos = row["Position (bp)"]

    if current_chr != chr_:
        if current_chr is not None:
            loci.append((current_chr, start, end))
        current_chr = chr_
        start = pos
        end = pos
    elif pos - end <= WINDOW:
        end = pos
    else:
        loci.append((current_chr, start, end))
        start = pos
        end = pos

if current_chr is not None:
    loci.append((current_chr, start, end))

print("Independent modifier loci (LD-pruned):", len(loci))
loci

Independent modifier loci (LD-pruned): 7


[(7, 425446, 465447),
 (7, 479589, 479589),
 (7, 509101, 509101),
 (7, 527159, 532771),
 (7, 557597, 557597),
 (7, 576468, 590527),
 (14, 478696, 484899)]

## Summarize LD-pruned Modifier Loci

We assign each modifier SNP to a locus ID,
then summarize signal strength per locus.

In [8]:
# Assign locus IDs
locus_ids = []
for chr_, start, end in loci:
    locus_ids.append({"Chromosome": chr_, "Start": start, "End": end})

locus_df = pd.DataFrame(locus_ids).reset_index().rename(columns={"index": "MOD_LOCUS_ID"})

# Map SNPs to loci
def assign_locus(chr_, pos):
    for _, row in locus_df.iterrows():
        if row["Chromosome"] == chr_ and row["Start"] <= pos <= row["End"]:
            return row["MOD_LOCUS_ID"]
    return np.nan

mod_pos["MOD_LOCUS_ID"] = mod_pos.apply(
    lambda r: assign_locus(r["Chromosome"], r["Position (bp)"]),
    axis=1
)

# Merge locus IDs into epi_clean
epi_clean = epi_clean.merge(
    mod_pos[["MOD_SNP", "MOD_LOCUS_ID"]],
    on="MOD_SNP",
    how="left"
)

locus_summary = (
    epi_clean
    .groupby("MOD_LOCUS_ID")
    .agg(
        n_interactions=("P", "count"),
        min_p=("P", "min"),
        max_abs_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values("n_interactions", ascending=False)
)

locus_summary

,n_interactions,min_p,max_abs_beta
MOD_LOCUS_ID,,,
0.0,1143,1.228000e-122,0.031940
6.0,715,9.999000e-31,0.019578
5.0,508,4.866000e-72,0.024969
3.0,254,2.861000e-99,0.028988
1.0,127,3.436000e-126,0.032397
2.0,127,3.256000e-109,0.030293
4.0,127,2.936000e-83,0.026760


## Map LD-pruned Locus IDs Back to Genomic Coordinates

This lets us interpret which biological region each locus corresponds to.

In [9]:
locus_df

,MOD_LOCUS_ID,Chromosome,Start,End
0,0,7,425446,465447
1,1,7,479589,479589
2,2,7,509101,509101
3,3,7,527159,532771
4,4,7,557597,557597
5,5,7,576468,590527
6,6,14,478696,484899


# LD-Pruned × QTL Epistasis Analysis — Key Conclusions So Far

We performed two epistasis scans:

1. **All SNPs × QTL SNPs**
2. **LD-pruned SNPs × QTL SNPs**

The LD-pruned analysis serves as a structural robustness test to ensure that
epistatic architecture is not an artifact of dense LD structure.

---

## LD-Pruned Scan Overview

After genome-wide Bonferroni correction and enforcing exactly one QTL SNP per pair:

- **Significant QTL × LD-pruned interactions:** 11,180  
- **Unique QTL SNPs involved:** 512  
- **Unique modifier SNPs involved:** 162  
- **QTL × QTL artifacts removed:** 628  

This is a drastic reduction from the full scan,
yet the core structure remains intact.

---

## Modifier Architecture (LD-Pruned)

Clustering modifier SNPs within 10 kb windows revealed:

**7 independent modifier loci**

- 6 loci on Chromosome 7
- 1 locus on Chromosome 14

Critically:

- The **Chromosome 7 region (~530 kb; MTL1 locus)** remains present.
- The **Chromosome 14 trafficking/RHO2 region** remains present.
- These are the same dominant regions identified in the full genome scan.

---

## Strength of Effects

Maximum |β_int| values remain high (~0.03),
comparable to the all-SNP scan.

This indicates:

- The signal is not driven by LD redundancy.
- Independent SNP anchors still detect strong epistasis.
- The architecture survives independence filtering.

---

## Structural Interpretation

The LD-pruned results confirm that:

- Epistasis is **modular**, not diffuse.
- The architecture is **biologically structured**, not random.
- There are at least two interacting QTL modules:
  - ERG11 / ergosterol pathway region
  - RHO2 / trafficking region
- These interact with a Chromosome 7 modifier block centered around the **MTL1 cell-wall stress sensor region**.

---

## Interim Biological Model

Drug perturbation → ergosterol pathway  
↓  
Membrane / cell wall stress  
↓  
MTL1 sensing and signaling  
↓  
Context-dependent modulation of resistance phenotype  

This structure is preserved under LD pruning,
strongly supporting that the epistasis is genuine biological interaction,
not statistical artifact.

---

Next step:
Validate whether **directionality of interaction effects**
(positive vs negative β_int)
remains modular across QTL intervals in the LD-pruned analysis.

## Check Directionality of Epistasis in the LD-Pruned Analysis

We test whether the **sign structure** we saw in the all×QTL scan is preserved here.

Approach:
1. Assign each **QTL_SNP** in `epi_clean` to a **QTL interval_id** using its genomic position.
2. Summarize interaction signs (β_int > 0 vs β_int < 0) per interval.
3. Identify intervals with near-perfect sign separation (strong evidence of modular biology).

In [10]:
 # Code Cell — Map QTL_SNPs to interval_id and summarize sign of β_int (LD-pruned)

# 0) Safety checks
for df_name, df in [("epi_clean", epi_clean), ("qtl_df", qtl_df), ("snp_info", snp_info)]:
    if df is None or len(df) == 0:
        raise ValueError(f"{df_name} is missing/empty")

# 1) Get positions for QTL_SNPs that actually appear in epi_clean
qtl_snps_pos = (
    pd.DataFrame({"QTL_SNP": epi_clean["QTL_SNP"].unique()})
    .merge(
        snp_info[["SNP", "Chromosome", "Position (bp)"]],
        left_on="QTL_SNP",
        right_on="SNP",
        how="left"
    )
)

if qtl_snps_pos["Chromosome"].isna().any():
    missing = qtl_snps_pos.loc[qtl_snps_pos["Chromosome"].isna(), "QTL_SNP"].head(10).tolist()
    raise ValueError(f"Some QTL_SNPs missing from snp_info. Example: {missing}")

# 2) For each QTL_SNP, find which interval it belongs to
rows = []
qtl_df_int = qtl_df.copy()
qtl_df_int["chrom"] = qtl_df_int["chrom"].astype(int)
qtl_df_int["start"] = qtl_df_int["start"].astype(int)
qtl_df_int["end"]   = qtl_df_int["end"].astype(int)

for _, srow in qtl_snps_pos.iterrows():
    snp_id = srow["QTL_SNP"]
    chr_ = int(srow["Chromosome"])
    pos = int(srow["Position (bp)"])

    hits = qtl_df_int.loc[
        (qtl_df_int["chrom"] == chr_) &
        (qtl_df_int["start"] <= pos) &
        (qtl_df_int["end"] >= pos),
        ["interval_id"]
    ]

    for _, h in hits.iterrows():
        rows.append({"QTL_SNP": snp_id, "interval_id": int(h["interval_id"])})

map_df_ld = pd.DataFrame(rows).drop_duplicates()

print("Mapped QTL_SNPs to intervals:", map_df_ld["QTL_SNP"].nunique())
print("Total SNP→interval mappings:", map_df_ld.shape[0])

# 3) Merge interval_id onto epi_clean (this merge is small: ~512 SNPs mapping)
epi_iv_ld = epi_clean.merge(map_df_ld, on="QTL_SNP", how="left")

na_ct = epi_iv_ld["interval_id"].isna().sum()
print("Interactions with missing interval_id:", na_ct)
if na_ct > 0:
    display(epi_iv_ld.loc[epi_iv_ld["interval_id"].isna(), ["QTL_SNP"]].drop_duplicates().head(10))

# 4) Sign summary per interval_id
sign_table_ld = (
    epi_iv_ld
    .groupby("interval_id")["BETA_INT"]
    .agg(
        negative=lambda x: (x < 0).sum(),
        positive=lambda x: (x > 0).sum(),
        total="count"
    )
    .sort_values("total", ascending=False)
)

sign_table_ld["frac_positive"] = sign_table_ld["positive"] / sign_table_ld["total"]
sign_table_ld["frac_negative"] = sign_table_ld["negative"] / sign_table_ld["total"]

sign_table_ld.head(25)

Mapped QTL_SNPs to intervals: 512
Total SNP→interval mappings: 512
Interactions with missing interval_id: 0


,negative,positive,total,frac_positive,frac_negative
interval_id,,,,,
34738,3105,3316,6421,0.516430,0.483570
16910,3162,1110,4272,0.259831,0.740169
37622,0,446,446,1.000000,0.000000
15201,0,41,41,1.000000,0.000000


## Summarize Effect Size Strength per QTL Interval (LD-Pruned)

We compute:
- Number of interactions
- Minimum P-value
- Maximum |β|
per QTL interval.

In [11]:
interval_strength_ld = (
    epi_iv_ld
    .groupby("interval_id")
    .agg(
        n_interactions=("P", "count"),
        min_p=("P", "min"),
        max_abs_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values("n_interactions", ascending=False)
)

interval_strength_ld

,n_interactions,min_p,max_abs_beta
interval_id,,,
34738,6421,3.436000e-126,0.032397
16910,4272,5.036000e-32,0.017595
37622,446,1.359000e-10,0.008784
15201,41,5.168000e-10,0.008721


## Save standardized outputs for downstream figure generation

This saves all key result tables in a structured directory:

tables/
    {TEST_TYPE}/
        {DRUG}/

Where:
- TEST_TYPE = "All vs QTL" or "LDpruned vs QTL"
- DRUG = "FLU" or "PULV"

These files are used exclusively by 05_figures.ipynb.

In [13]:
from pathlib import Path

# -----------------------------
# CONFIG (edit these two)
# -----------------------------
DRUG = "PULV"                 # "FLU" or "PULV"
TEST_TYPE = "LDpruned vs QTL"     

BASE = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis")
OUT_DIR = BASE / "tables" / TEST_TYPE / DRUG
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helper: pick the right object name automatically
# -----------------------------
def pick_first_existing(candidates, label):
    for name in candidates:
        if name in globals() and globals()[name] is not None:
            return name, globals()[name]
    raise NameError(f"None of these {label} variables exist: {candidates}")

# -----------------------------
# 1) Clean interaction table
# -----------------------------
name_epi, df_epi = pick_first_existing(["epi_clean"], "epi_clean")
df_epi.to_csv(OUT_DIR / "epi_clean.tsv", sep="\t", index=False)

# -----------------------------
# 2) Interval-level summary (04B uses interval_summary)
# -----------------------------
name_int, df_int = pick_first_existing(
    ["interval_summary", "interval_strength", "interval_strength_ld", "interval_strength_all"],
    "interval summary"
)
df_int.to_csv(OUT_DIR / "interval_summary.tsv", sep="\t")

# -----------------------------
# 3) Modifier locus summary (if present)
# -----------------------------
# (04B might not have locus_summary depending on how far you went)
try:
    name_locus, df_locus = pick_first_existing(
        ["locus_summary", "locus_summary_ld", "modifier_locus_summary"],
        "locus summary"
    )
    df_locus.to_csv(OUT_DIR / "modifier_locus_summary.tsv", sep="\t")
    locus_saved = True
except NameError:
    locus_saved = False

# -----------------------------
# 4) Sign summary (if present)
# -----------------------------
try:
    name_sign, df_sign = pick_first_existing(
        ["sign_table", "sign_table_ld", "sign_summary", "sign_table_all"],
        "sign table"
    )
    df_sign.to_csv(OUT_DIR / "sign_summary.tsv", sep="\t")
    sign_saved = True
except NameError:
    sign_saved = False

print("Saved outputs to:", OUT_DIR)
print(" - epi_clean.tsv  (from:", name_epi, ")")
print(" - interval_summary.tsv (from:", name_int, ")")
print(" - modifier_locus_summary.tsv:", "saved" if locus_saved else "NOT FOUND in notebook (ok for now)")
print(" - sign_summary.tsv:", "saved" if sign_saved else "NOT FOUND in notebook (ok for now)")

print("\nFiles now in OUT_DIR:")
for f in sorted(OUT_DIR.iterdir()):
    print(" -", f.name)

Saved outputs to: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/tables/LDpruned vs QTL/PULV
 - epi_clean.tsv  (from: epi_clean )
 - interval_summary.tsv (from: interval_strength_ld )
 - modifier_locus_summary.tsv: saved
 - sign_summary.tsv: saved

Files now in OUT_DIR:
 - epi_clean.tsv
 - interval_strength.tsv
 - interval_summary.tsv
 - modifier_locus_summary.tsv
 - sign_summary.tsv
